# Test du modèle sur des chants gascon (duo Corbefin-Marsac, La Man Encantada)
## 1. Imports

In [1]:
import os
from pathlib import Path

Import from scripts

In [2]:
from models.w2v2p import load_model, predict_simple
from data.get_data import get_phonemes_from_json
from data.form_dataset import make_dataset
from data.get_closest import closest
from data.cleaning import clean_list
from evaluation.metrics import get_per_nltk
from evaluation.pipeline import evaluate_dataset
from evaluation.alignment import alignement_ajuste

/home/manon/Documents/Master/M2/stage/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Loading paths

In [3]:
base_path = Path("..")
song_path = base_path / "data/chants/"

## 3. Loading model

In [4]:
processor, model = load_model("facebook/wav2vec2-xlsr-53-espeak-cv-ft")

Loading weights: 100%|██████████| 424/424 [00:00<00:00, 742.35it/s, Materializing param=wav2vec2.masked_spec_embed]                                            


## 4. Loading data

In [5]:
sentences_cm = []
for part in ["part1", "part2", "part3"]:
    sent_path = song_path / f"congo_{part}.txt"
    with open(sent_path, "r", encoding="utf-8") as file:
        sentences_cm.append(file.read())

sentences_man_encantada = []
man_encantada = {
    'man_encantada': ['part1'],
    'nei_tant': ['part1', 'part2', 'part3', 'part4'],
}
for name, part in man_encantada.items():
    for p in part:
        sent_path = song_path / f"{name}_{p}.txt"
        with open(sent_path, "r", encoding="utf-8") as file:
            sentences_man_encantada.append(file.read())

The Man Encantada corpus contains songs with an instrumental accompaniment - man_vocal contains the same songs without the instrumental accompaniment

In [6]:
y_test = []
y_test_cm = get_phonemes_from_json(base_path / "scripts_propres/phonemes_congo.json")
y_test_man = get_phonemes_from_json(base_path / "scripts_propres/phonemes_man_encantada.json")

for y in [y_test_cm, y_test_man, y_test_man]:
    y_test.extend(y)

X_test_cm = []
X_test_man = [os.path.join(song_path,"man_encantada_part1.mp3")]
X_test_man_vocal = [os.path.join(song_path, "man_encantada_vocal_part1.mp3")]
X_test = []

for part in ["part1", "part2", "part3"]:
    X_test_cm.append(os.path.join(song_path, f"congo_{part}.mp3"))

for part in ["part1", "part2", "part3", "part4"]:
    X_test_man.append(os.path.join(song_path, f"nei_tant_{part}.mp3"))

for part in ["part1", "part2", "part3", "part4"]:
    X_test_man_vocal.append(os.path.join(song_path, f"nei_tant_vocal_{part}.mp3"))

for x in [X_test_cm, X_test_man, X_test_man_vocal]:
    X_test.extend(x)

## 5. Build dataset

In [7]:
cm_dataset = make_dataset(X_test_cm, y_test_cm)
man_dataset = make_dataset(X_test_man, y_test_man)
man_vocal_dataset = make_dataset(X_test_man_vocal, y_test_man)
song_dataset = make_dataset(X_test, y_test)

## 6. Run inference

In [8]:
per_total, aligned_per_total = evaluate_dataset(
    song_dataset,
    X_test,
    y_test,
    model,
    processor,
    predict_simple,
    clean_list,
    closest,
    get_per_nltk,
    alignement_ajuste,
)

per_cm, aligned_per_cm = evaluate_dataset(
    cm_dataset,
    X_test_cm,
    y_test_cm,
    model,
    processor,
    predict_simple,
    clean_list,
    closest,
    get_per_nltk,
    alignement_ajuste,
)

per_man, aligned_per_man = evaluate_dataset(
    man_dataset,
    X_test_man,
    y_test_man,
    model,
    processor,
    predict_simple,
    clean_list,
    closest,
    get_per_nltk,
    alignement_ajuste,
)

per_man_vocal, aligned_per_man_vocal = evaluate_dataset(
    man_vocal_dataset,
    X_test_man_vocal,
    y_test_man,
    model,
    processor,
    predict_simple,
    clean_list,
    closest,
    get_per_nltk,
    alignement_ajuste,
)

100%|██████████| 178/178 [00:00<00:00, 569.97it/s]


In [9]:
print(f"PER on the whole corpus : {per_total} | Aligned PER on the whole corpus : {aligned_per_total}")
print(f"PER Corbefin-Marsac : {per_cm} | Aligned PER Corbefin-Marsac : {aligned_per_cm}")
print(f"PER Man Encantada : {per_man} | Aligned PER Man Encantada : {aligned_per_man}")
print(f"PER Man Encantada vocal : {per_man_vocal} | Aligned PER Man Encantada vocal : {aligned_per_man_vocal}")

PER on the whole corpus : 44.72% | Aligned PER on the whole corpus : 43.65%
PER Corbefin-Marsac : 60.50% | Aligned PER Corbefin-Marsac : 62.50%
PER Man Encantada : 41.68% | Aligned PER Man Encantada : 40.62%
PER Man Encantada vocal : 38.30% | Aligned PER Man Encantada vocal : 36.46%
